# End-to-End RAG with Structured Excel + Unstructured PDF using Google Gemini

## What you will build

A complete **Retrieval-Augmented Generation (RAG)** pipeline that can answer questions from:

- **Structured data:** an Excel sheet containing SAP incident records.
- **Unstructured data:** a PDF containing support and escalation policies.
- **Google Gemini:** used for text generation and embeddings.
- **LangChain:** used for documents, chunking, prompting, and model integration.
- **ChromaDB:** used as the local vector database.

### Final architecture

```text
                         ┌───────────────────────────┐
                         │       User Question       │
                         └─────────────┬─────────────┘
                                       │
                                       ▼
                         ┌───────────────────────────┐
                         │   Gemini Query Embedding  │
                         └─────────────┬─────────────┘
                                       │
                                       ▼
           ┌──────────────────────────────────────────────────┐
           │                 Chroma Vector Store              │
           │                                                  │
           │  Excel row-documents  +  PDF text chunks        │
           └──────────────────────┬───────────────────────────┘
                                  │ Top-k relevant chunks
                                  ▼
                         ┌───────────────────────────┐
                         │  Grounded Prompt Context  │
                         └─────────────┬─────────────┘
                                       │
                                       ▼
                         ┌───────────────────────────┐
                         │     Google Gemini LLM     │
                         └─────────────┬─────────────┘
                                       │
                                       ▼
                         ┌───────────────────────────┐
                         │ Answer + Source Metadata  │
                         └───────────────────────────┘
```

## Learning objectives

By the end of this notebook, you will understand:

1. Why Excel and PDF data should not always be chunked in the same way.
2. How to convert Excel rows into LangChain `Document` objects.
3. How to load and chunk PDF pages.
4. How embeddings convert chunks into searchable vectors.
5. How Chroma retrieves semantically relevant chunks.
6. How to pass retrieved context to Gemini.
7. How to build one reusable `ask_rag()` function with source-aware answers.

## 1. Install the required libraries

Run the following cell once in Google Colab.

> After installation, Colab may occasionally ask you to restart the runtime if a preinstalled dependency was replaced.

In [ ]:
!pip install -qU \
    langchain \
    langchain-core \
    langchain-community \
    langchain-google-genai \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    pypdf \
    pandas \
    openpyxl \
    reportlab

## 2. Import libraries

We use `pandas` for Excel, `PyPDFLoader` for PDF extraction, Gemini for embeddings and generation, and Chroma for vector search.

In [ ]:
import os
import getpass
from pathlib import Path
from typing import Optional

import pandas as pd

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from langchain_chroma import Chroma

print("Imports completed successfully.")

## 3. Configure your Google AI API key

### Recommended option in Google Colab

Open the **Secrets** panel in Colab and create a secret named:

```text
GOOGLE_API_KEY
```

The cell below first tries to read that secret. If it cannot find it, it securely asks you to paste the key using `getpass`.

> Your key is not printed by this notebook.

In [ ]:
try:
    from google.colab import userdata
    google_api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    google_api_key = None

if not google_api_key:
    google_api_key = getpass.getpass("Enter your Google AI API key: ")

os.environ["GOOGLE_API_KEY"] = google_api_key
print("Google AI API key configured.")

## 4. Choose the Gemini models

This tutorial defaults to:

- **Generation model:** `gemini-2.5-flash`
- **Embedding model:** `gemini-embedding-001`

These defaults are suitable for a text-based RAG tutorial using Excel records and text extracted from PDFs.

In [ ]:
GENERATION_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL = "gemini-embedding-001"

print("Generation model:", GENERATION_MODEL)
print("Embedding model:", EMBEDDING_MODEL)

# Part A — Prepare the data

You can use the built-in demo files or upload your own `.xlsx` and `.pdf` files.

For the first run, keep:

```python
USE_DEMO_FILES = True
```

This makes the notebook fully self-contained.

## 5. Create sample Excel and PDF files

The demo Excel contains structured SAP incident records. The demo PDF contains unstructured support policy text. This supports Excel-only, PDF-only, and cross-source RAG questions.

In [ ]:
USE_DEMO_FILES = True

if USE_DEMO_FILES:
    incident_data = [
        {
            "incident_id": "INC-1001",
            "module": "SAP MM",
            "priority": "P2",
            "category": "Invoice Management",
            "issue_summary": "Supplier invoice blocked because of a purchase-order price variance.",
            "resolution": "Corrected the purchase-order condition record and reprocessed the invoice.",
            "owner_team": "Procure-to-Pay Support",
            "resolution_time_hours": 3.5,
        },
        {
            "incident_id": "INC-1002",
            "module": "SAP HANA",
            "priority": "P1",
            "category": "Database Availability",
            "issue_summary": "HANA database became unavailable after severe memory exhaustion.",
            "resolution": "Freed memory, restarted the affected service, and tuned allocation limits.",
            "owner_team": "Database Platform Team",
            "resolution_time_hours": 5.0,
        },
        {
            "incident_id": "INC-1003",
            "module": "SAP SD",
            "priority": "P3",
            "category": "Pricing",
            "issue_summary": "Sales order showed an incorrect discount because of pricing procedure configuration.",
            "resolution": "Corrected the condition sequence and repriced the sales order.",
            "owner_team": "Order-to-Cash Support",
            "resolution_time_hours": 1.2,
        },
        {
            "incident_id": "INC-1004",
            "module": "SAP BTP",
            "priority": "P1",
            "category": "Connectivity",
            "issue_summary": "Destination authentication failure prevented a cloud application from reaching the backend.",
            "resolution": "Renewed credentials, corrected destination properties, and validated connectivity.",
            "owner_team": "BTP Platform Team",
            "resolution_time_hours": 7.5,
        },
        {
            "incident_id": "INC-1005",
            "module": "SAP SuccessFactors",
            "priority": "P2",
            "category": "Integration",
            "issue_summary": "Employee replication was delayed because a scheduled integration job failed.",
            "resolution": "Fixed the mapping error and reran the failed integration job.",
            "owner_team": "HR Integration Team",
            "resolution_time_hours": 4.0,
        },
        {
            "incident_id": "INC-1006",
            "module": "SAP HANA",
            "priority": "P1",
            "category": "Performance",
            "issue_summary": "A long-running savepoint caused severe performance degradation for critical business transactions.",
            "resolution": "Identified the savepoint bottleneck, corrected storage pressure, and optimized the workload.",
            "owner_team": "Database Platform Team",
            "resolution_time_hours": 9.0,
        },
    ]

    demo_excel_path = "/content/sap_incidents_demo.xlsx"
    pd.DataFrame(incident_data).to_excel(demo_excel_path, index=False)

    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak

    demo_pdf_path = "/content/sap_support_policy_demo.pdf"
    styles = getSampleStyleSheet()
    story = []

    policy_sections = [
        (
            "SAP Enterprise Support and Escalation Policy",
            """This policy defines the support response and escalation expectations for production incidents. It applies to SAP application, integration, cloud platform, and database incidents handled by the enterprise support organization.""",
        ),
        (
            "P1 — Critical Business Outage",
            """A P1 incident is a critical production outage or severe degradation that blocks essential business operations. The support team must start immediate triage, assign an incident commander, and provide stakeholder updates every 15 minutes. Functional and platform escalation must occur within 30 minutes. If the incident remains unresolved after 2 hours, executive escalation is required.""",
        ),
        (
            "P2 — High Business Impact",
            """A P2 incident has high business impact but a complete business outage has not occurred. Initial triage should start within 1 hour. Status updates should be provided at least every 2 hours. If no stable workaround or resolution is available after 4 hours, the issue should be escalated to the responsible domain lead.""",
        ),
        (
            "P3 and P4 — Medium and Low Priority",
            """P3 incidents should be triaged within 4 business hours and handled through the normal support queue. P4 requests may include minor defects, information requests, or low-impact improvements and should be planned according to service capacity.""",
        ),
        (
            "Cross-Team Escalation Principles",
            """When an incident spans multiple technology domains, one incident commander must own coordination. Database, application, integration, security, and cloud platform teams should share evidence through a common incident timeline. Escalation should be based on business impact, elapsed resolution time, and absence of a reliable workaround.""",
        ),
        (
            "Post-Incident Review",
            """Every P1 incident should have a post-incident review that records the timeline, root cause, business impact, corrective action, preventive action, and accountable owners. Repeated incident patterns should be converted into automation, monitoring, knowledge articles, or permanent engineering fixes.""",
        ),
    ]

    for i, (title, body) in enumerate(policy_sections):
        story.append(Paragraph(title, styles["Heading1"]))
        story.append(Spacer(1, 12))
        story.append(Paragraph(body.strip(), styles["BodyText"]))
        story.append(Spacer(1, 18))
        if i in {1, 3}:
            story.append(PageBreak())

    pdf = SimpleDocTemplate(demo_pdf_path, pagesize=A4)
    pdf.build(story)

    print("Demo files created:")
    print("Excel:", demo_excel_path)
    print("PDF:  ", demo_pdf_path)
else:
    print("Demo-file creation skipped because USE_DEMO_FILES=False.")

## 6. Select demo files or upload your own files

To upload your own data, set `USE_DEMO_FILES = False` above, run the next cell, and upload one Excel file plus one PDF file.

In [ ]:
if USE_DEMO_FILES:
    excel_path = demo_excel_path
    pdf_path = demo_pdf_path
else:
    from google.colab import files

    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())

    excel_candidates = [
        name for name in uploaded_names
        if name.lower().endswith((".xlsx", ".xls"))
    ]
    pdf_candidates = [
        name for name in uploaded_names
        if name.lower().endswith(".pdf")
    ]

    if not excel_candidates:
        raise ValueError("No Excel file found. Upload a .xlsx or .xls file.")
    if not pdf_candidates:
        raise ValueError("No PDF file found. Upload a .pdf file.")

    excel_path = f"/content/{excel_candidates[0]}"
    pdf_path = f"/content/{pdf_candidates[0]}"

print("Selected Excel:", excel_path)
print("Selected PDF:  ", pdf_path)

# Part B — Load and chunk the Excel data

## Why structured data should be chunked differently

For an Excel table, one row usually represents one business entity or transaction: one incident, customer, order, or product. Blind character chunking can break the meaning of a record.

### Better strategy

1. Read the sheet with pandas.
2. Convert every row into readable `column: value` text.
3. Preserve row number and business identifier as metadata.
4. Split only unusually long rows.

This is **row-aware chunking**.

## 7. Convert Excel rows into LangChain documents

In [ ]:
def excel_to_documents(
    file_path: str,
    sheet_name=0,
) -> list[Document]:
    """Convert each Excel row into one LangChain Document."""

    workbook = pd.ExcelFile(file_path)

    if isinstance(sheet_name, int):
        actual_sheet_name = workbook.sheet_names[sheet_name]
    else:
        actual_sheet_name = sheet_name

    df = pd.read_excel(file_path, sheet_name=actual_sheet_name)
    documents = []

    for dataframe_index, row in df.iterrows():
        clean_record = {
            str(column): value
            for column, value in row.to_dict().items()
            if pd.notna(value)
        }

        row_text = "\n".join(
            f"{column}: {value}"
            for column, value in clean_record.items()
        )

        metadata = {
            "source_type": "excel",
            "source": Path(file_path).name,
            "sheet_name": actual_sheet_name,
            "row_number": int(dataframe_index + 2),
            "record_id": str(
                clean_record.get(
                    "incident_id",
                    clean_record.get("id", "")
                )
            ),
        }

        documents.append(
            Document(
                page_content=row_text,
                metadata=metadata,
            )
        )

    return documents


excel_documents = excel_to_documents(excel_path)

print("Excel row-documents:", len(excel_documents))
print("\nFirst Excel document:\n")
print(excel_documents[0].page_content)
print("\nMetadata:")
print(excel_documents[0].metadata)

## 8. Apply a safety splitter to long Excel rows

Most ordinary rows stay intact. This splitter only helps when a row contains large text fields such as issue descriptions, resolution notes, or customer comments.

In [ ]:
excel_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
)

excel_chunks = excel_splitter.split_documents(excel_documents)

for i, doc in enumerate(excel_chunks):
    doc.metadata["chunk_id"] = f"excel-{i + 1}"

print("Excel chunks after safety splitting:", len(excel_chunks))

# Part C — Load and chunk the unstructured PDF

A PDF is unstructured text. A page may contain several ideas, and a single idea may continue across paragraphs. `RecursiveCharacterTextSplitter` is a practical general-purpose default because it tries natural separators first and progressively falls back to smaller ones.

## 9. Load PDF pages

In [ ]:
pdf_loader = PyPDFLoader(pdf_path)
pdf_pages = pdf_loader.load()

print("PDF pages loaded:", len(pdf_pages))
print("\nSample extracted text:\n")
print(pdf_pages[0].page_content[:1000])

## 10. Add cleaner PDF metadata and split into chunks

We use `chunk_size=1000` and `chunk_overlap=150`. The overlap helps retain context near chunk boundaries.

In [ ]:
for page_doc in pdf_pages:
    page_doc.metadata["source_type"] = "pdf"
    page_doc.metadata["source"] = Path(pdf_path).name

    if "page" in page_doc.metadata:
        page_doc.metadata["page_number"] = int(page_doc.metadata["page"]) + 1

pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

pdf_chunks = pdf_splitter.split_documents(pdf_pages)

for i, doc in enumerate(pdf_chunks):
    doc.metadata["chunk_id"] = f"pdf-{i + 1}"

print("PDF chunks:", len(pdf_chunks))
print("\nFirst PDF chunk:\n")
print(pdf_chunks[0].page_content[:1000])
print("\nMetadata:")
print(pdf_chunks[0].metadata)

# Part D — Combine both data sources

Excel rows are now semantic documents/chunks, and PDF pages have become overlapping text chunks. Both can be indexed together.

In [ ]:
all_chunks = excel_chunks + pdf_chunks

print("Total chunks:", len(all_chunks))
print("Excel chunks:", len(excel_chunks))
print("PDF chunks:  ", len(pdf_chunks))

source_counts = pd.Series(
    [doc.metadata["source_type"] for doc in all_chunks]
).value_counts()

print("\nChunk count by source type:")
print(source_counts)

# Part E — Create embeddings and the vector store

## What is an embedding?

An embedding converts text into a numerical vector. Texts with similar meaning should be closer together in vector space, allowing semantic retrieval even when exact words differ.

## 11. Initialize Gemini embeddings

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
)

# Small connectivity test
test_vector = embeddings.embed_query(
    "SAP HANA production database incident"
)

print("Embedding created.")
print("Vector dimensions:", len(test_vector))
print("First 5 values:", test_vector[:5])

## 12. Create the Chroma vector store

`Chroma.from_documents()` embeds each chunk and stores its vector together with text and metadata.

In [ ]:
vector_store = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="excel_pdf_gemini_rag",
)

print("Vector store created successfully.")
print("Indexed chunks:", len(all_chunks))

# Part F — Test semantic retrieval before using the LLM

A strong RAG system should inspect retrieval quality first. The LLM can only answer correctly if the right evidence is retrieved.

## 13. Inspect retrieved chunks

In [ ]:
def show_retrieved_documents(documents: list[Document]) -> None:
    """Pretty-print retrieved documents and metadata."""
    for rank, doc in enumerate(documents, start=1):
        metadata = doc.metadata
        source_type = metadata.get("source_type", "unknown")
        source = metadata.get("source", "unknown")

        if source_type == "excel":
            location = f"row {metadata.get('row_number', '?')}"
        elif source_type == "pdf":
            location = f"page {metadata.get('page_number', '?')}"
        else:
            location = "location unknown"

        print("=" * 90)
        print(f"Rank {rank} | {source_type.upper()} | {source} | {location}")
        print("-" * 90)
        print(doc.page_content[:1200])
        print()


retrieval_question = (
    "Which P1 SAP HANA incident took the longest to resolve, "
    "and what escalation policy applies to a P1 incident?"
)

retrieved_docs = vector_store.similarity_search(
    retrieval_question,
    k=5,
)

show_retrieved_documents(retrieved_docs)

# Part G — Build the grounded Gemini RAG pipeline

The final pipeline:

1. Embeds the user question.
2. Retrieves the most relevant Excel/PDF chunks.
3. Formats the chunks with source metadata.
4. Asks Gemini to answer only from that context.

## 14. Initialize the Gemini chat model

In [ ]:
llm = ChatGoogleGenerativeAI(
    model=GENERATION_MODEL,
    temperature=0,
    max_retries=2,
)

print("Gemini chat model initialized:", GENERATION_MODEL)

## 15. Create a grounded RAG prompt

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a grounded enterprise RAG assistant.

Rules:
1. Answer only from the retrieved context.
2. Never invent facts not supported by the context.
3. If the answer is unavailable, say exactly:
   "I don't know based on the provided Excel and PDF files."
4. Distinguish Excel facts from PDF facts.
5. Use exact incident IDs, priorities, row numbers, page numbers, and values when available.
6. When possible, cite evidence inline like:
   [Excel: filename, row 4]
   [PDF: filename, page 2]
7. For cross-source questions, explicitly connect the structured record with the relevant policy text.
            """.strip(),
        ),
        (
            "human",
            """
Retrieved context:
{context}

Question:
{question}

Provide a concise but complete answer grounded only in the retrieved context.
            """.strip(),
        ),
    ]
)

## 16. Helper functions for context formatting and source display

In [ ]:
def document_source_label(doc: Document) -> str:
    metadata = doc.metadata
    source_type = metadata.get("source_type", "unknown")
    source = metadata.get("source", "unknown")

    if source_type == "excel":
        return (
            f"Excel: {source}, "
            f"sheet={metadata.get('sheet_name', '?')}, "
            f"row={metadata.get('row_number', '?')}, "
            f"record_id={metadata.get('record_id', '')}"
        )

    if source_type == "pdf":
        return (
            f"PDF: {source}, "
            f"page={metadata.get('page_number', '?')}"
        )

    return f"Source: {source}"


def format_context(documents: list[Document]) -> str:
    context_parts = []

    for index, doc in enumerate(documents, start=1):
        label = document_source_label(doc)
        context_parts.append(
            f"""
[Retrieved Source {index}]
{label}
Content:
{doc.page_content}
            """.strip()
        )

    return "\n\n".join(context_parts)


def unique_source_labels(documents: list[Document]) -> list[str]:
    labels = []
    seen = set()

    for doc in documents:
        label = document_source_label(doc)
        if label not in seen:
            labels.append(label)
            seen.add(label)

    return labels

## 17. Build the reusable `ask_rag()` function

Parameters:

- `question`: natural-language question.
- `k`: number of chunks to retrieve.
- `source_type`: optional filter: `"excel"` or `"pdf"`.

Without a filter, retrieval searches both sources.

In [ ]:
def ask_rag(
    question: str,
    k: int = 5,
    source_type: Optional[str] = None,
    show_sources: bool = True,
) -> str:
    """Retrieve relevant documents and generate a grounded Gemini answer."""

    if source_type not in {None, "excel", "pdf"}:
        raise ValueError("source_type must be None, 'excel', or 'pdf'.")

    if source_type is None:
        documents = vector_store.similarity_search(
            question,
            k=k,
        )
    else:
        documents = vector_store.similarity_search(
            question,
            k=k,
            filter={"source_type": source_type},
        )

    context = format_context(documents)

    messages = rag_prompt.format_messages(
        context=context,
        question=question,
    )

    response = llm.invoke(messages)
    answer = response.content

    print("QUESTION")
    print(question)
    print("\nANSWER")
    print(answer)

    if show_sources:
        print("\nRETRIEVED SOURCES")
        for index, source in enumerate(
            unique_source_labels(documents),
            start=1,
        ):
            print(f"{index}. {source}")

    return answer

# Part H — Ask questions

## 18. Excel-only question

In [ ]:
ask_rag(
    question=(
        "Which P1 incident took the longest to resolve? "
        "Give the incident ID, SAP module, issue summary, "
        "resolution, and resolution time."
    ),
    k=4,
    source_type="excel",
)

## 19. PDF-only question

In [ ]:
ask_rag(
    question=(
        "What are the escalation requirements and stakeholder "
        "update cadence for a P1 critical business outage?"
    ),
    k=4,
    source_type="pdf",
)

## 20. Cross-source RAG question

This is the key example: retrieve an incident from Excel, retrieve the matching policy from PDF, and connect both sources in one answer.

In [ ]:
ask_rag(
    question=(
        "Among the P1 SAP HANA incidents in the Excel data, "
        "which one took the longest to resolve? According to the PDF policy, "
        "what escalation actions should have applied during that incident?"
    ),
    k=6,
)

## 21. Ask your own question

In [ ]:
my_question = """
Which repeated patterns in the incident data might deserve automation,
monitoring, knowledge articles, or permanent engineering fixes according
to the support policy?
""".strip()

ask_rag(
    question=my_question,
    k=6,
)

# Part I — Understand the complete RAG flow

When you call `ask_rag("your question")`:

### Step 1 — Query embedding
Gemini converts the question into a numeric vector.

### Step 2 — Semantic retrieval
Chroma compares the query vector with stored chunk vectors and returns the most relevant chunks.

### Step 3 — Context construction
The notebook builds context containing retrieved text plus file, row, page, and record metadata.

### Step 4 — Grounded generation
Gemini receives the retrieved evidence and question. The system prompt prohibits unsupported facts.

### Step 5 — Source transparency
The notebook prints the source locations returned by retrieval.

# Part J — Important design decisions

## Why not convert the entire Excel sheet into one giant text block?
A question about one record could retrieve a huge amount of unrelated data. Row-level documents preserve record boundaries.

## Why use a recursive splitter for PDF?
PDF text is unstructured. Recursive splitting is a practical general-purpose approach that tries to keep natural text units together.

## Why use overlap?
A small overlap reduces the chance of losing context at chunk boundaries.

## Why store metadata?
Metadata enables traceability, source filtering, auditing, and human-readable citations.

## Why inspect retrieval before generation?
Many apparent LLM problems are actually retrieval problems. Always verify whether correct evidence is retrieved.

# Part K — Production improvements

For a production-grade system, consider:

1. **Multi-sheet Excel support** — load every sheet and preserve `sheet_name` metadata.
2. **Schema-aware serialization** — normalize dates, numbers, IDs, and important business fields.
3. **Hybrid search** — combine vector retrieval with keyword/BM25 search for IDs and codes.
4. **Reranking** — retrieve a wider candidate set, then rerank before generation.
5. **Persistent vector storage** — persist Chroma or use a managed vector database.
6. **Incremental indexing** — re-embed only changed records.
7. **RAG evaluation** — measure retrieval relevance, groundedness, correctness, and citation quality.
8. **Access control** — filter data according to user authorization.
9. **Table-native reasoning** — combine RAG with pandas/SQL tools for calculations, aggregation, joins, and exact filtering.
10. **OCR for scanned PDFs** — `PyPDFLoader` works best with text-based PDFs; image-only scans need OCR or document understanding.

# Final summary

You built a RAG system that combines:

```text
Structured Excel records
        +
Unstructured PDF text
        ↓
LangChain Documents
        ↓
Source-specific chunking
        ↓
Gemini embeddings
        ↓
Chroma vector database
        ↓
Semantic retrieval
        ↓
Grounded prompt
        ↓
Google Gemini answer
```

## Key takeaway

**Do not chunk every data source the same way.**

- Excel benefits from **record-aware / row-aware chunking**.
- PDF text benefits from **recursive text splitting with overlap**.
- Both can be indexed in one vector store and retrieved together.

# Official references

- Google Gemini API models: https://ai.google.dev/gemini-api/docs/models
- Google Gemini embeddings: https://ai.google.dev/gemini-api/docs/embeddings
- LangChain Gemini chat integration: https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
- LangChain Gemini embeddings integration: https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai
- LangChain text splitters: https://docs.langchain.com/oss/python/integrations/splitters
- LangChain semantic search / RAG tutorial: https://docs.langchain.com/oss/python/langchain/knowledge-base
- LangChain Chroma integration: https://docs.langchain.com/oss/python/integrations/vectorstores/chroma